# Week 5 - DS Automation Assignment - Michael Easton

## AI Use Disclosure:
To my knowledge I did not use AI to research, create, or complete any portion of this assignment; however, due to AI's integration into general internet searching functionality and many websites, it is possible debugging efforts may have inadvertantly included the use of AI. My intent is to primarily use source documentation for python, pandas, and other modules first before seeking answers through broader internet search.

# Instructions

## Task 1
Using our prepared churn data from week 2:
- use pycaret to find an ML algorithm that performs best on the data
    - Choose a metric you think is best to use for finding the best model; by default, it is accuracy but it could be AUC, precision, recall, etc. The week 3 FTE has some information on these different metrics.
- save the model to disk
- create a Python script/file/module with a function that takes a pandas dataframe as an input and returns the probability of churn for each row in the dataframe
    - your Python file/function should print out the predictions for new data (new_churn_data.csv)
    - the true values for the new data are [1, 0, 0, 1, 0] if you're interested
- test your Python module and function with the new data, new_churn_data.csv
- write a short summary of the process and results at the end of this notebook
- upload this Jupyter Notebook and Python file to a Github repository, and turn in a link to the repository in the week 5 assignment dropbox

Additional challenges:
- return the probability of churn for each new prediction, and the percentile where that prediction is in the distribution of probability predictions from the training dataset (e.g. a high probability of churn like 0.78 might be at the 90th percentile)
- use other autoML packages, such as TPOT, H2O, MLBox, etc, and compare performance and features with pycaret
- create a class in your Python module to hold the functions that you created
- accept user input to specify a file using a tool such as Python's `input()` function, the `click` package for command-line arguments, or a GUI
- Use the unmodified churn data (new_unmodified_churn_data.csv) in your Python script. This will require adding the same preprocessing steps from week 2 since this data is like the original unmodified dataset from week 1.


## Task 2: Critical Reflection
What are your observations based on the analyses you had performed? Is there one model that seems particularly useful for predicting churn. Why or why not?

# Task 1  Programming Steps

In [1]:
# Troubleshooting dependencies, attempting to create a new environment here.
#%conda create -n msds python=3.10.14 -y
#%conda init msds
#%conda activate msds

#%conda install pycaret
#python --version

#Setup a new environment for Python 3.11.9 for pycarat
#%conda install -c conda-forge pycaret -y
#%conda install scipy==1.11.4     #Noted that pycaret had trouble importing interp from scipy after 1.11.4 
#https://stackoverflow.com/questions/78008260/importerror-unable-to-import-interp-from-scipy-module
#%conda install scikit-learn==0.23.2 --force-reinstall

#Final Note: Creating a new environment via the command prompt vs. anaconda navigator seemed to work.
# After creation it only involved the installation of pycarat, but that seems like it was enough.

In [2]:
import pandas as pd

## Task 1.1 Pre-Processsed Data

In [3]:
#Importing the Churn Data from Week 2
df = pd.read_csv("churn_data.csv", index_col='customerID')
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7043 entries, 7590-VHVEG to 3186-AJIEK
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   tenure          7043 non-null   int64  
 1   PhoneService    7043 non-null   object 
 2   Contract        7043 non-null   object 
 3   PaymentMethod   7043 non-null   object 
 4   MonthlyCharges  7043 non-null   float64
 5   TotalCharges    7032 non-null   float64
 6   Churn           7043 non-null   object 
dtypes: float64(2), int64(1), object(4)
memory usage: 440.2+ KB


In [4]:
new_df = pd.read_csv("new_churn_data.csv", index_col='customerID')
new_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, 9305-CKSKC to 6348-TACGU
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   tenure             5 non-null      int64  
 1   PhoneService       5 non-null      int64  
 2   Contract           5 non-null      int64  
 3   PaymentMethod      5 non-null      int64  
 4   MonthlyCharges     5 non-null      float64
 5   TotalCharges       5 non-null      float64
 6   charge_per_tenure  5 non-null      float64
dtypes: float64(3), int64(4)
memory usage: 320.0+ bytes


In [5]:
# Need to create the "Charge_Per_Tenure" column category to match the datasets and make the rest numerical.
df['charge_per_tenure'] = df['TotalCharges']/df['tenure']
df['Contract'] = df['Contract'].replace({'Month-to-month': 0, 'One year': 1, 'Two year': 2})
df['PaymentMethod'] = df['PaymentMethod'].replace({'Credit card (automatic)': 0, 'Bank transfer (automatic)':1, 'Mailed check':2, 'Electronic check':3})
df['PhoneService'] = df['PhoneService'].replace({'Yes':1,'No':0})
df['Churn'] = df['Churn'].replace({'Yes':1,'No':0})
df = df[df['TotalCharges'].isna()==False]

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 7590-VHVEG to 3186-AJIEK
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   tenure             7032 non-null   int64  
 1   PhoneService       7032 non-null   int64  
 2   Contract           7032 non-null   int64  
 3   PaymentMethod      7032 non-null   int64  
 4   MonthlyCharges     7032 non-null   float64
 5   TotalCharges       7032 non-null   float64
 6   Churn              7032 non-null   int64  
 7   charge_per_tenure  7032 non-null   float64
dtypes: float64(3), int64(5)
memory usage: 494.4+ KB


### 1.1.1 PyCaret Pre-Processed

In [6]:
from pycaret.classification import setup, compare_models, predict_model, save_model, load_model

In [34]:
dfList = list(df.columns)
dfList.remove("Churn")
dfList
pc_automl = setup(data = df, target = "Churn", preprocess = False)
#, numeric_features = dfList, fold_shuffle = True)

,Description,Value
0,Session id,7794
1,Target,Churn
2,Target type,Binary
3,Original data shape,"(7032, 8)"
4,Transformed data shape,"(7032, 8)"
5,Transformed train set shape,"(4922, 8)"
6,Transformed test set shape,"(2110, 8)"
7,Numeric features,7


Metric Explanations for Future Reference:

- Accuracy - % of how many Non-Churn cases were actually predicted as Non-Churn
- Recall - % of how many Churn cases were predicted to Churn
- Precision - % of how many Churn & Non-Churn cases were correctly predicted
- AUC - Area under the curve of ROC (True Positive vs. False Positive rate) - High shows catching all positives and having no false negatives
- F1 - Harmonic Mean of Recall & Precision (High is good) - 
- Kappa - Cohen's Kappa compares accuracy against random chance and is a potentially better accuracy measure in unbalanced data (like this data)
- MCC (Matthews Correlation Coefficient) - If all four measures of the confusion matrix are high, MCC will be high.
- TT(Sec) - Relative time to run the model

Ref: 
- https://www.drivedatascience.com/model-evaluation-confusion-matrix-precision-recall-f1-roc-auc-cross-validation/
- https://www.statisticshowto.com/matthews-correlation-coefficient/
- https://www.geeksforgeeks.org/machine-learning/f1-score-in-machine-learning/
- https://karam-nus.github.io/metrics/docs/metrics/classification/cohens_kappa

In [35]:
recall_model = compare_models(sort = 'Recall')
recall_model

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
qda,Quadratic Discriminant Analysis,0.7477,0.8217,0.7393,0.5186,0.6093,0.4315,0.4464,0.0060
nb,Naive Bayes,0.7438,0.8162,0.7293,0.5138,0.6025,0.4221,0.4364,0.1920
lr,Logistic Regression,0.7944,0.8359,0.5252,0.6392,0.5760,0.4420,0.4462,0.2430
lda,Linear Discriminant Analysis,0.7887,0.8238,0.4984,0.6301,0.5562,0.4201,0.4253,0.0050
ada,Ada Boost Classifier,0.7875,0.8345,0.4947,0.6280,0.5530,0.4162,0.4216,0.0190
svm,SVM - Linear Kernel,0.7629,0.7686,0.4945,0.5830,0.5117,0.3623,0.3780,0.0050
dt,Decision Tree Classifier,0.7269,0.6553,0.4885,0.4869,0.4866,0.3010,0.3016,0.0060
gbc,Gradient Boosting Classifier,0.7899,0.8377,0.4847,0.6394,0.5510,0.4173,0.4244,0.0490
rf,Random Forest Classifier,0.7720,0.8086,0.4755,0.5880,0.5244,0.3771,0.3816,0.0400
et,Extra Trees Classifier,0.7546,0.7851,0.4595,0.5468,0.4984,0.3378,0.3406,0.0260


QuadraticDiscriminantAnalysis(priors=None, reg_param=0.0,
                              store_covariance=False, tol=0.0001)

In [53]:
from pycaret.classification import create_model
QDA_setup = setup(data=df, target='Churn', preprocess = False)
QDA_model = create_model('qda')

,Description,Value
0,Session id,511
1,Target,Churn
2,Target type,Binary
3,Original data shape,"(7032, 8)"
4,Transformed data shape,"(7032, 8)"
5,Transformed train set shape,"(4922, 8)"
6,Transformed test set shape,"(2110, 8)"
7,Numeric features,7


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7221,0.8074,0.7252,0.4847,0.5810,0.3852,0.4027
1,0.7931,0.8543,0.7863,0.5819,0.6688,0.5232,0.5358
2,0.7337,0.8054,0.7000,0.4973,0.5815,0.3943,0.4067
3,0.7663,0.8496,0.7769,0.5401,0.6372,0.4729,0.4899
4,0.7337,0.7985,0.6870,0.5000,0.5788,0.3911,0.4017
5,0.7256,0.7968,0.7099,0.4895,0.5794,0.3859,0.4006
6,0.7724,0.8217,0.7099,0.5569,0.6242,0.4643,0.4713
7,0.7398,0.8147,0.7176,0.5081,0.5949,0.4115,0.4248
8,0.7398,0.8182,0.7328,0.5079,0.6000,0.4165,0.4318


In [54]:
save_model(QDA_model, 'QDA_Churn')

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('placeholder', None),
                 ('trained_model',
                  QuadraticDiscriminantAnalysis(priors=None, reg_param=0.0,
                                                store_covariance=False,
                                                tol=0.0001))],
          verbose=False),
 'QDA_Churn.pkl')

In [55]:
kappa_model = compare_models(sort ='Kappa')
kappa_model

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.7980,0.8374,0.5276,0.6479,0.5812,0.4501,0.4544,0.0120
gbc,Gradient Boosting Classifier,0.7993,0.8407,0.5000,0.6621,0.5694,0.4419,0.4495,0.0500
qda,Quadratic Discriminant Analysis,0.7481,0.8196,0.7256,0.5195,0.6052,0.4278,0.4408,0.0050
ada,Ada Boost Classifier,0.7932,0.8370,0.4939,0.6463,0.5593,0.4274,0.4344,0.0190
lda,Linear Discriminant Analysis,0.7893,0.8232,0.4931,0.6347,0.5543,0.4193,0.4255,0.0060
nb,Naive Bayes,0.7430,0.8162,0.7179,0.5122,0.5975,0.4165,0.4297,0.0050
ridge,Ridge Classifier,0.7911,0.8231,0.4465,0.6597,0.5318,0.4038,0.4170,0.0060
rf,Random Forest Classifier,0.7759,0.8044,0.4802,0.5982,0.5322,0.3873,0.3916,0.0390
et,Extra Trees Classifier,0.7623,0.7854,0.4764,0.5629,0.5156,0.3596,0.3621,0.0270
knn,K Neighbors Classifier,0.7657,0.7467,0.4473,0.5760,0.5031,0.3532,0.3582,0.0130


LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=1000,
                   multi_class='auto', n_jobs=None, penalty='l2',
                   random_state=511, solver='lbfgs', tol=0.0001, verbose=0,
                   warm_start=False)

In [39]:
LR_setup = setup(data=df, target='Churn')
LR_model = create_model('lr')

,Description,Value
0,Session id,3940
1,Target,Churn
2,Target type,Binary
3,Original data shape,"(7032, 8)"
4,Transformed data shape,"(7032, 8)"
5,Transformed train set shape,"(4922, 8)"
6,Transformed test set shape,"(2110, 8)"
7,Numeric features,7
8,Preprocess,True
9,Imputation type,simple


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.8316,0.8682,0.5573,0.7449,0.6376,0.5309,0.5403
1,0.7911,0.8352,0.4427,0.6591,0.5297,0.4020,0.4151
2,0.8171,0.8551,0.5769,0.6818,0.6250,0.5051,0.5082
3,0.7744,0.8101,0.5462,0.5772,0.5613,0.4096,0.4099
4,0.7927,0.8377,0.5267,0.6330,0.5750,0.4394,0.4427
5,0.7541,0.8006,0.4504,0.5463,0.4937,0.3333,0.3360
6,0.7988,0.8340,0.5725,0.6356,0.6024,0.4682,0.4694
7,0.7927,0.8489,0.5191,0.6355,0.5714,0.4365,0.4404
8,0.8110,0.8625,0.5573,0.6759,0.6109,0.4876,0.4915


In [40]:
df.iloc[-2:-1]

,tenure,PhoneService,Contract,PaymentMethod,MonthlyCharges,TotalCharges,Churn,charge_per_tenure
customerID,,,,,,,,
8361-LTMKD,4,1,0,2,74.4,306.6,1,76.65


In [41]:
predict_model(LR_model, df.iloc[-2:-1])

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Logistic Regression,1.0000,0,1.0000,1.0000,1.0000,nan,0.0000


,tenure,PhoneService,Contract,PaymentMethod,MonthlyCharges,TotalCharges,charge_per_tenure,Churn,prediction_label,prediction_score
customerID,,,,,,,,,,
8361-LTMKD,4,1,0,2,74.400002,306.600006,76.650002,1,1,0.5779


In [56]:
save_model(LR_model, 'LR_Churn')

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('placeholder', None),
                 ('trained_model',
                  LogisticRegression(C=1.0, class_weight=None, dual=False,
                                     fit_intercept=True, intercept_scaling=1,
                                     l1_ratio=None, max_iter=1000,
                                     multi_class='auto', n_jobs=None,
                                     penalty='l2', random_state=3940,
                                     solver='lbfgs', tol=0.0001, verbose=0,
                                     warm_start=False))],
          verbose=False),
 'LR_Churn.pkl')

In [43]:
import pickle
with open('LR_Churn_Model.pk', 'wb') as f:
    pickle.dump(kappa_model, f)

In [44]:
with open('LR_Churn_Model.pk', 'rb') as f:
    loaded_model = pickle.load(f)

In [45]:
nChurnDF = df.iloc[-2:-1].copy()
nChurnDF.drop('Churn', axis=1, inplace=True)
loaded_model.predict(nChurnDF)

array([1], dtype=int8)

In [46]:
loaded_nb = load_model("LR_Churn")

Transformation Pipeline and Model Successfully Loaded


In [47]:
predict_model(loaded_nb, nChurnDF)

,tenure,PhoneService,Contract,PaymentMethod,MonthlyCharges,TotalCharges,charge_per_tenure,prediction_label,prediction_score
customerID,,,,,,,,,
8361-LTMKD,4,1,0,2,74.400002,306.600006,76.650002,1,0.5779


In [48]:
predict_model(loaded_model)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Logistic Regression,0.7839,0.8254,0.4973,0.6159,0.5503,0.4102,0.4142


,tenure,PhoneService,Contract,PaymentMethod,MonthlyCharges,TotalCharges,charge_per_tenure,Churn,prediction_label,prediction_score
customerID,,,,,,,,,,
8778-LMWTJ,9,1,0,3,74.849998,708.200012,78.688889,0,1,0.5706
1193-RTSLK,38,1,0,0,83.900002,3233.600098,85.094734,1,0,0.7308
1707-HABPF,46,1,1,1,91.300003,4126.350098,89.703262,0,0,0.8580
2507-QZPQS,15,1,0,3,95.699997,1451.099976,96.739998,0,1,0.6736
5429-LWCMV,12,1,0,3,89.150002,1057.550049,88.129166,0,1,0.6475
...,...,...,...,...,...,...,...,...,...,...
0526-SXDJP,72,0,2,1,42.099998,2962.000000,41.138889,0,0,0.9928
3011-WQKSZ,19,1,0,3,20.000000,377.549988,19.871052,0,0,0.8649
1471-GIQKQ,1,1,0,3,49.950001,49.950001,49.950001,0,0,0.5338


In [57]:
from IPython.display import Code

Code('ChurnModel.py')

# -*- coding: utf-8 -*-
"""
Spyder Editor

This is a temporary script file.
"""

class ChurnModel:
    """
    Modified functions from Week 5 FTE, expanded to be a class and include additional
    information.
    Load Data, Make a Prediction (Pre-Processed), Present Prediction and
    probabilities.
    Defaults to Pre-Proccessed Data (PPD) = True. Currently only works with
    Pre-Processed Data.
    """
    
    def __init__(self, filepath, bestmodel, PPD = True, last_prediction = None):
        self.filepath = filepath
        self.bestmodel = bestmodel
        self.PPD = PPD
        self.last_prediction = last_prediction

    def develop_predictions(self):
        import pandas as pd
        from pycaret.classification import predict_model, load_model
        
        def load_data(filepath):
            """
            Loads Churn data into a DataFrame from a string filepath.
            """
            df = pd.read_csv(filepath, index_col='customerID')
            return df
        
        def make_predictions(self, mpdf):
            """
            Uses the pycaret best model to make predictions on data in the df dataframe.
            """
            model = load_model(self.bestmodel)
            mppredictions = predict_model(model, data=mpdf)

            # Check the column names
            print(mppredictions.columns)
            
            # Rename 'prediction_label' to 'Churn_prediction' if it exists
            if 'prediction_label' in mppredictions.columns:
                mppredictions.rename(columns={'prediction_label': 'Churn_Prediction', 'prediction_score':'Churn_Probability'}, inplace=True)
                
                # Replace values in the new column
                mppredictions['Churn_Prediction'].replace({1: 'Churn', 0: 'No Churn'}, inplace=True)
                
                #Add a new Column with the Prediction Percentages
                if self.PPD == True:                
                    return mppredictions.drop(['tenure','PhoneService','Contract','PaymentMethod','MonthlyCharges','TotalCharges','charge_per_tenure'], axis=1)
                else:
                    return mppredictions.drop(['tenure','PhoneService','Contract','PaymentMethod','MonthlyCharges','TotalCharges'], axis=1)
            else:
                raise KeyError("The 'prediction_label' column was not found in the predictions DataFrame")

        
        df = load_data(self.filepath)
        self.last_prediction = make_predictions(self,df)
    
    def show_prediction(self):
        print("Prediction:")
        print(self.last_prediction)

In [58]:
%run ChurnModel.py

In [59]:
CM = ChurnModel('new_churn_data.csv','LR_Churn')
CM.develop_predictions()
CM.show_prediction()

Transformation Pipeline and Model Successfully Loaded


Index(['tenure', 'PhoneService', 'Contract', 'PaymentMethod', 'MonthlyCharges',
       'TotalCharges', 'charge_per_tenure', 'prediction_label',
       'prediction_score'],
      dtype='object')
Prediction:
           Churn_Prediction  Churn_Probability
customerID                                    
9305-CKSKC            Churn             0.5449
1452-KNGVK         No Churn             0.5586
6723-OKKJM         No Churn             0.9439
7832-POPKP         No Churn             0.8364
6348-TACGU         No Churn             0.5522


[1, 0, 0, 1, 0]

In [60]:
DM = ChurnModel('new_churn_data.csv','QDA_Churn')
DM.develop_predictions()
DM.show_prediction()

Transformation Pipeline and Model Successfully Loaded


Index(['tenure', 'PhoneService', 'Contract', 'PaymentMethod', 'MonthlyCharges',
       'TotalCharges', 'charge_per_tenure', 'prediction_label',
       'prediction_score'],
      dtype='object')
Prediction:
           Churn_Prediction  Churn_Probability
customerID                                    
9305-CKSKC            Churn                1.0
1452-KNGVK            Churn                1.0
6723-OKKJM            Churn                1.0
7832-POPKP            Churn                1.0
6348-TACGU            Churn                1.0


## Task 1.2 Non-Processed Data

In [26]:
u_df = pd.read_csv("churn_data.csv", index_col='customerID')
u_df.info()
u_newdf = pd.read_csv("new_churn_data_unmodified.csv", index_col = 'customerID')
u_newdf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7043 entries, 7590-VHVEG to 3186-AJIEK
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   tenure          7043 non-null   int64  
 1   PhoneService    7043 non-null   object 
 2   Contract        7043 non-null   object 
 3   PaymentMethod   7043 non-null   object 
 4   MonthlyCharges  7043 non-null   float64
 5   TotalCharges    7032 non-null   float64
 6   Churn           7043 non-null   object 
dtypes: float64(2), int64(1), object(4)
memory usage: 440.2+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, 9305-CKSKC to 6348-TACGU
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   tenure          5 non-null      int64  
 1   PhoneService    5 non-null      object 
 2   Contract        5 non-null      object 
 3   PaymentMethod   5 non-null      object 
 4   MonthlyCharges  5 non-null 

### 1.2.1 PyCaret Non-Pre-Processed

In [27]:
u_dfList = list(u_df.columns)
u_dfList.remove("Churn")
u_dfList
u_pc_automl = setup(data = u_df, target = "Churn")

,Description,Value
0,Session id,7792
1,Target,Churn
2,Target type,Binary
3,Target mapping,"No: 0, Yes: 1"
4,Original data shape,"(7043, 7)"
5,Transformed data shape,"(7043, 12)"
6,Transformed train set shape,"(4930, 12)"
7,Transformed test set shape,"(2113, 12)"
8,Numeric features,3
9,Categorical features,3


In [28]:
u_model = compare_models(sort = 'Recall')
u_model

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.8002,0.8424,0.8002,0.7910,0.7930,0.4545,0.4592,0.0470
ada,Ada Boost Classifier,0.7963,0.8415,0.7963,0.7876,0.7892,0.4450,0.4501,0.0260
lr,Logistic Regression,0.7961,0.8401,0.7961,0.7879,0.7896,0.4471,0.4514,0.0280
ridge,Ridge Classifier,0.7939,0.8302,0.7939,0.7821,0.7829,0.4228,0.4319,0.0120
lda,Linear Discriminant Analysis,0.7929,0.8302,0.7929,0.7851,0.7872,0.4421,0.4454,0.0110
rf,Random Forest Classifier,0.7736,0.8095,0.7736,0.7653,0.7678,0.3926,0.3953,0.0490
et,Extra Trees Classifier,0.7613,0.7895,0.7613,0.7542,0.7569,0.3670,0.3683,0.0370
knn,K Neighbors Classifier,0.7590,0.7511,0.7590,0.7454,0.7494,0.3383,0.3426,0.0210
svm,SVM - Linear Kernel,0.7438,0.7685,0.7438,0.7634,0.7406,0.3640,0.3765,0.0120
dummy,Dummy Classifier,0.7347,0.5000,0.7347,0.5398,0.6223,0.0000,0.0000,0.0120


GradientBoostingClassifier(ccp_alpha=0.0, criterion='friedman_mse', init=None,
                           learning_rate=0.1, loss='log_loss', max_depth=3,
                           max_features=None, max_leaf_nodes=None,
                           min_impurity_decrease=0.0, min_samples_leaf=1,
                           min_samples_split=2, min_weight_fraction_leaf=0.0,
                           n_estimators=100, n_iter_no_change=None,
                           random_state=7792, subsample=1.0, tol=0.0001,
                           validation_fraction=0.1, verbose=0,
                           warm_start=False)

In [29]:
ADA_setup = setup(data=u_df, target='Churn')
ADA_model = create_model('ada')

,Description,Value
0,Session id,3117
1,Target,Churn
2,Target type,Binary
3,Target mapping,"No: 0, Yes: 1"
4,Original data shape,"(7043, 7)"
5,Transformed data shape,"(7043, 12)"
6,Transformed train set shape,"(4930, 12)"
7,Transformed test set shape,"(2113, 12)"
8,Numeric features,3
9,Categorical features,3


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7809,0.8308,0.7809,0.7656,0.7676,0.3775,0.3868
1,0.8154,0.8323,0.8154,0.8059,0.8015,0.4652,0.4824
2,0.7972,0.8420,0.7972,0.7848,0.7850,0.4272,0.4377
3,0.8073,0.8666,0.8073,0.7967,0.7969,0.4602,0.4696
4,0.7890,0.8305,0.7890,0.7758,0.7772,0.4075,0.4163
5,0.7566,0.8031,0.7566,0.7420,0.7466,0.3306,0.3347
6,0.7972,0.8522,0.7972,0.7866,0.7888,0.4422,0.4477
7,0.8012,0.8383,0.8012,0.7926,0.7950,0.4618,0.4652
8,0.7830,0.8177,0.7830,0.7690,0.7712,0.3920,0.4000


In [30]:
save_model(ADA_model, 'ADA_Model')

Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('label_encoding',
                  TransformerWrapperWithInverse(exclude=None, include=None,
                                                transformer=LabelEncoder())),
                 ('numerical_imputer',
                  TransformerWrapper(exclude=None,
                                     include=['tenure', 'MonthlyCharges',
                                              'TotalCharges'],
                                     transformer=SimpleImputer(add_indicator=False,
                                                               copy=True,
                                                               fill_value=None,
                                                               keep_empty_features=False,...
                                     include=['Contract', 'PaymentMethod'],
                                     transformer=OneHotEncoder(cols=['Contract',
                                                       

In [31]:
loaded_npp = load_model("ADA_Model")
u_nChurnDF = df.iloc[-2:-1].copy()
u_nChurnDF.drop('Churn', axis=1, inplace=True)
predict_model(loaded_npp, u_nChurnDF)

Transformation Pipeline and Model Successfully Loaded


,tenure,PhoneService,Contract,PaymentMethod,MonthlyCharges,TotalCharges,charge_per_tenure,prediction_label,prediction_score
customerID,,,,,,,,,
8361-LTMKD,4,1,0,2,74.400002,306.600006,76.650002,No,0.5064


In [32]:
AM = ChurnModel("new_churn_data_unmodified.csv","ADA_Model", PPD =False)

In [33]:
AM.develop_predictions()
AM.show_prediction()

Transformation Pipeline and Model Successfully Loaded


Index(['tenure', 'PhoneService', 'Contract', 'PaymentMethod', 'MonthlyCharges',
       'TotalCharges', 'prediction_label', 'prediction_score'],
      dtype='object')
Prediction:
           Churn_Prediction  Churn_Probability
customerID                                    
9305-CKSKC              Yes             0.5048
1452-KNGVK               No             0.5066
6723-OKKJM               No             0.5117
7832-POPKP              Yes             0.5008
6348-TACGU               No             0.5169


# Summary

This week, I practiced using the AutoML feature of the module pycarat to compare and create a model, save it, then load it back in to predict Churn on a set of new data. For the first task I attempted to pre-process churn_data; however, I quickly realized that the churn_data and new_churn_data featured different features that needed to be remedied. In converting categorical data to numerical, I also realized that I did not know with what methods the new_churn_data was converted to numerical data. Comparing values between these two spreadsheets and new_churn_data_unmodified it appeared that at least Payment Type didn't follow a discernable pattern. 

With this possible issue noted, I still progressed in creating models with this cleaned up churn_data, focusing on the measures of Recall,as this might estimate the Churned accounts better, and Kappa, which is a similar measure to accuracy with improvements in how it handles unbalanced data like the churn set. In doing this I followed the process chain for both models (except only creating one pickle file based on the Logistic Regression model). This yielded a Linear Regression Model and a Quadratic Discriminat Analysis Model. 

A third model was created using unprocessed data. I had to do some minor clean-up but in general it was a simpler process. This yielded a Gradient Boosting Classifier model.

With these three models, I created Python script to more generalize input and prediction of new data. This came in the form of Python class object called ChurnModel which takes in the new data set, the name of the saved model, and whether or not the data was Pre-Processed/Pre-Cleaned. Running the develop_prediction method in this object creates a set of prediction results for the input data that is stored in the Model object as a last_prediction. In the future it may be useful to have this last_prediction variable store a longer history to monitor the performance of the model over time.

The results of each model are below:

**Cleaned Data**
| Quadratic Discriminant Analysis | Churn Actual | Prediction | Probability |
| --- | --- | --- | --- |
| --- | 1 | Churn | 1.0 |
| --- | 0 | Churn | 1.0 |
| --- | 0 | Churn | 1.0 |
| --- | 1 | Churn | 1.0 |
| --- | 0 | Churn | 1.0 |

| Logistic Regression | Churn Actual | Prediction | Probability |
| --- | --- | --- | --- |
| --- | 1 | Churn | 0.5449 |
| --- | 0 | No Churn | 0.5586 |
| --- | 0 | No Churn | 0.9439 |
| --- | 1 | No Churn | 0.8364 |
| --- | 0 | No Churn | 0.5522 |

**Raw Data**
| Gradient Boosting Classifier | Churn Actual | Prediction | Probability |
| --- | --- | --- | --- |
| --- | 1 | Churn | 0.5048 |
| --- | 0 | No Churn | 0.5066 |
| --- | 0 | No Churn | 0.5117 |
| --- | 1 | Churn | 0.5008 |
| --- | 0 | No Churn | 0.5169 |

Gradient Boosting Classifier got 100% on it's prediction of the data; however, looking at the probabilities it seems as if it'd be close to a 50-50 shot. Logistic Regression, presented probabilities that were more "certain" of their result but it did not catch the 2nd Churn case. It instead featured a high probability that case was a No Churn Case. That said, it appears Logistic Regression might fit the data better, but I would need to confirm the way in which the new_churn_data was cleaned to ensure it matches how the training data was cleaned. There was a difference in the categorical variables that could present a problem that the Gradient Boosting Classifier didn't need to accomodate.